In [1]:
import json
import pandas as pd
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent

PAPERS_FILE = PROJECT_ROOT / "data" / "raw" / "papers.jsonl"
CITATIONS_FILE = PROJECT_ROOT / "data" / "raw" / "citations.csv"

print("Papers file:")
print(PAPERS_FILE)

print("\nCitations file:")
print(CITATIONS_FILE)

print("\nPapers exists:", PAPERS_FILE.exists())
print("Citations exists:", CITATIONS_FILE.exists())

Papers file:
c:\Users\Shubham\OneDrive\Desktop\MajorProject\data\raw\papers.jsonl

Citations file:
c:\Users\Shubham\OneDrive\Desktop\MajorProject\data\raw\citations.csv

Papers exists: True
Citations exists: True


In [2]:
papers = []

with open(PAPERS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        papers.append(json.loads(line))

print("Number of papers:", len(papers))

Number of papers: 5000


In [3]:
papers_df = pd.DataFrame(papers)

papers_df.head()

,paperId,title,abstract,authors,year,venue,citationCount,fieldsOfStudy,publicationDate,openAccessPdf
0,00000c33779acab142af6c7a6dae8b36fac0805d,Insights into Household Electric Vehicle Charg...,In the era of burgeoning electric vehicle (EV)...,"[{'authorId': '66925799', 'name': 'Ahmad Almag...",2024.0,Energies,23,None,2024-02-16,{'url': 'https://www.mdpi.com/1996-1073/17/4/9...
1,0000238f07f151172cf2602588ba762b55c8464b,Personalized prediction of response to smartph...,Meditation apps are popular and may reduce psy...,"[{'authorId': '2247677350', 'name': 'Christian...",2021.0,,1,[Medicine],2021-07-29,{'url': 'https://doi.org/10.31234/osf.io/drqa4...
2,0000315635be19f6278dbc72597b3065fac405f0,Abstractive text summarization of low-resource...,Background Humans must be able to cope with th...,"[{'authorId': '2201143605', 'name': 'Nida Shaf...",2023.0,PeerJ Computer Science,30,"[Computer Science, Medicine]",2023-01-13,{'url': 'https://doi.org/10.7717/peerj-cs.1176...
3,00005d68c6c7eb4d3c27da8242a30b9a498f991e,Detection of DDoS Attacks on Clouds Computing ...,The growing number of cloud-based services has...,"[{'authorId': '2085549408', 'name': 'Iehab Alr...",2023.0,International Conference on Communication and ...,7,None,2023-06-19,"{'url': '', 'status': 'CLOSED', 'license': Non..."
4,0000627a402141da113dfb8148711fecfcb4fdd1,Design of Network Intrusion Detection Model Ba...,The traditional machine learning model cannot ...,"[{'authorId': '2058044165', 'name': 'Quan Wen'}]",2022.0,Security and Communication Networks,1,None,2022-07-07,{'url': 'https://downloads.hindawi.com/journal...


In [4]:
papers_df.isnull().sum()

paperId               0
title                 0
abstract              0
authors               0
year                  8
venue                 0
citationCount         0
fieldsOfStudy      2028
publicationDate     328
openAccessPdf         0
dtype: int64

In [5]:
print("Missing titles:",
      papers_df["title"].isna().sum())

print("Missing abstracts:",
      papers_df["abstract"].isna().sum())

print("Missing years:",
      papers_df["year"].isna().sum())

Missing titles: 0
Missing abstracts: 0
Missing years: 8


In [6]:
citations_df = pd.read_csv(CITATIONS_FILE)

print("Number of citation rows:", len(citations_df))

citations_df.head()

Number of citation rows: 103496


,source_paper,target_paper,relationship
0,00000c33779acab142af6c7a6dae8b36fac0805d,9357a46f47c6f2bea9761cbcd79ddab4e350d557,reference
1,00000c33779acab142af6c7a6dae8b36fac0805d,c59c06c8a44acbda74667e76459b2df8dae90273,reference
2,00000c33779acab142af6c7a6dae8b36fac0805d,286519b404064fe4e52f1dc5483596fe683aed03,reference
3,00000c33779acab142af6c7a6dae8b36fac0805d,49e3c326f67a5916bdb617f9399b21c16f9803ad,reference
4,00000c33779acab142af6c7a6dae8b36fac0805d,4bf43099563822c04eb58e6ac7c745680eafcbe7,reference


In [7]:
citations_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 103496 entries, 0 to 103495
Data columns (total 3 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   source_paper  103496 non-null  str  
 1   target_paper  103496 non-null  str  
 2   relationship  103496 non-null  str  
dtypes: str(3)
memory usage: 2.4 MB


In [8]:
print(
    "Duplicate paper IDs:",
    papers_df["paperId"].duplicated().sum()
)

Duplicate paper IDs: 0


In [9]:
paper_ids = set(papers_df["paperId"])

source_ids = set(citations_df["source_paper"])
target_ids = set(citations_df["target_paper"])

print("Papers:", len(paper_ids))

print("Unique citation sources:", len(source_ids))
print("Unique citation targets:", len(target_ids))

print(
    "Sources inside dataset:",
    len(source_ids & paper_ids)
)

print(
    "Targets inside dataset:",
    len(target_ids & paper_ids)
)

Papers: 5000
Unique citation sources: 3291
Unique citation targets: 88884
Sources inside dataset: 2573
Targets inside dataset: 119


In [10]:
graph_edges = citations_df[
    citations_df["source_paper"].isin(paper_ids) &
    citations_df["target_paper"].isin(paper_ids)
].copy()

print("Edges inside 5,000-paper dataset:",
      len(graph_edges))

graph_edges.head()

Edges inside 5,000-paper dataset: 107


,source_paper,target_paper,relationship
403,0001d28a5e0600e6b34b1fefcf4486f2c60a0a5d,0040fbc24133d560c7de163f4bd7a32e07f9fd1e,reference
759,0021e8add8de95c5a54e4527e5338a37fec93a09,0003e0ac0f8f4b50065f0e0c43b95896639fe2a3,citation
916,00044a527a0642b08f385cda845331f0fc8cbbfd,0075552c4336abfacda2c15c1ebc3507e3f45200,reference
1273,005d2c93fe65f39e2defe22e12253baac160207a,00044a527a0642b08f385cda845331f0fc8cbbfd,citation
1487,00059087c954c1af6ece33115315e3e0ecc2f2c2,00159a3f909534e328276ac4a4424a100802708c,reference


In [11]:
# Keep only citation edges where BOTH papers
# belong to our 5,000-paper dataset.

graph_edges = citations_df[
    citations_df["source_paper"].isin(paper_ids) &
    citations_df["target_paper"].isin(paper_ids)
].copy()

print("Total CSV edges:", len(citations_df))
print("Edges inside 5,000-paper dataset:", len(graph_edges))

print(
    "Unique source papers:",
    graph_edges["source_paper"].nunique()
)

print(
    "Unique target papers:",
    graph_edges["target_paper"].nunique()
)

Total CSV edges: 103496
Edges inside 5,000-paper dataset: 107
Unique source papers: 102
Unique target papers: 84


In [12]:
graph_edges.head(10)

,source_paper,target_paper,relationship
403,0001d28a5e0600e6b34b1fefcf4486f2c60a0a5d,0040fbc24133d560c7de163f4bd7a32e07f9fd1e,reference
759,0021e8add8de95c5a54e4527e5338a37fec93a09,0003e0ac0f8f4b50065f0e0c43b95896639fe2a3,citation
916,00044a527a0642b08f385cda845331f0fc8cbbfd,0075552c4336abfacda2c15c1ebc3507e3f45200,reference
1273,005d2c93fe65f39e2defe22e12253baac160207a,00044a527a0642b08f385cda845331f0fc8cbbfd,citation
1487,00059087c954c1af6ece33115315e3e0ecc2f2c2,00159a3f909534e328276ac4a4424a100802708c,reference
1730,00066a5fd641c90ead9f78a6b128d09a5edc35e9,0169e5c907b4b509e4d7d59c260849282e454277,reference
2936,000cd7ac3435007cca292fc8f1bac86b7a066467,00c7a21bd4d7c2c67ae54efbd2f6336cd5dc17e6,reference
5154,001555747a6afdf31245679a6588240887954691,00ef71115eac25c3b536e88bf95187db8bd16103,reference
5314,0016318d683dc3dab0512657df6e98016f2f64e0,0090023afc66cd2741568599057f4e82b566137c,reference
7314,002113c2a0d507ddfcadc5e8049af5cfeda91e24,006263988e0450f03f1a3a7796fa039363edfbc2,reference


In [13]:
graph_edges["relationship"].value_counts()

relationship
reference    105
citation       2
Name: count, dtype: int64

In [14]:
import networkx as nx

G = nx.DiGraph()

G.add_nodes_from(paper_ids)

G.add_edges_from(
    graph_edges[
        ["source_paper", "target_paper"]
    ].itertuples(
        index=False,
        name=None
    )
)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

print(
    "Nodes with at least one connection:",
    sum(
        1
        for node in G.nodes
        if G.degree(node) > 0
    )
)

print(
    "Isolated nodes:",
    sum(
        1
        for node in G.nodes
        if G.degree(node) == 0
    )
)

Nodes: 5000
Edges: 107
Nodes with at least one connection: 182
Isolated nodes: 4818


In [15]:
papers_df[
    papers_df["year"].isna()
][[
    "paperId",
    "title",
    "abstract"
]]

,paperId,title,abstract
831,005174832042a95c9bf40e959173fe67a76ecc5a,Variable-dependent predictability of coastal a...,Advances in environmental sensing technologies...
1839,00b6f6178a2f1acc2ba7f1c5a2d35dad930a9eca,Forecasting Hotel Daily Occupancy for High-Fre...,Accurate hotel daily occupancy forecasting is ...
1935,00c145f0c61e536b294ee456afc75cf9d78300f5,Pattern classification of time-series signals ...,The objective of this study is to assess the p...
2322,00e4be3f283f84bd49ab4254db18efdc10ab3fda,Brain functional fingerprinting predicts indiv...,Machine learning approaches provide convenient...
2750,010e1691d4b14900543cb0431714b3726521c663,and Consumption in,: The energy revolution and digital revolution...
2871,011956e32ecd20f62dfd52e68051473ce96af3e7,Trustworthy machine learning methods for the a...,Machine Learning (ML) is widely applied to aut...
3082,012f19a63153b5d02f7bec3b095b5e5d083e6a19,Quality of service compliance verification usi...,Service Level Agreements (SLAs) are formal con...
4422,01b1c931c2135d24ec634eb926ae96eb2d93c711,Bridging in-game behaviors and learning outcom...,Digital Game-Based Learning (DGBL) holds the p...
